# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravindidhananjana/Internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking / Scoring (Learning to Rank)

Why: The core goal is not just binary classification (declining vs. not declining), but ordering declining/low-performing pages so an editor can focus on the top 5–10 high-impact pages each week.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load dataset directly from raw GitHub URL so it works in Google Colab
url = "https://raw.githubusercontent.com/ravindidhananjana/Internship-ML/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Total pages/URLs in dataset: {len(df)}")

Total pages/URLs in dataset: 30000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/Outcome: A calculated priority score or rank based on observed historical drops in impressions/clicks, position tier, and search demand.

Label Source: Derived from observed outcomes in historical Search Console metrics (data/raw/content_refresh_anonymized.csv), rather than an arbitrary manual rule.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Calculate drops between previous 30 days and last 30 days
df['impressions_drop'] = df['impressions_prev_30d'] - df['impressions_last_30d']
df['clicks_drop'] = df['clicks_prev_30d'] - df['clicks_last_30d']

# Create a priority score combining impression & click drops
df['priority_score'] = (df['impressions_drop'] * 1) + (df['clicks_drop'] * 10)

df[['content_id', 'impressions_drop', 'clicks_drop', 'priority_score']].sort_values(
    by='priority_score', ascending=False
).head()

,content_id,impressions_drop,clicks_drop,priority_score
6653,content_5fe46e04994d,97995,30,98295
26844,content_8c19996aa890,71821,138,73201
21565,content_9532f197bbc8,64918,264,67558
13537,content_2c2606c5d176,59831,281,62641
15968,content_66b4046cc144,61895,11,62005


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: NDCG (Normalized Discounted Cumulative Gain) or Mean Average Precision (MAP) for ranking, or Precision@K (e.g., Precision@10).Target / Good Value: High precision in the top 10 recommended pages (e.g., $\ge 80\%$ of the top 10 suggested pages are truly high-impact candidates needing a refresh)

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Count pages experiencing a positive drop in clicks (declining performance)
declining_pages = df[df['clicks_drop'] > 0]
print(f"Declining pages needing review: {len(declining_pages)} out of {len(df)} ({len(declining_pages)/len(df):.1%})")

Declining pages needing review: 6806 out of 30000 (22.7%)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One URL / Page.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Unit of Analysis: 1 row = 1 piece of content / page (content_id)
print("Dataset Shape:", df.shape)
df[['content_id', 'impressions_drop', 'clicks_drop', 'position_tier', 'priority_score']].head()


Dataset Shape: (30000, 47)


,content_id,impressions_drop,clicks_drop,position_tier,priority_score
0,content_304f48230142,409,11,striking,519
1,content_a1fb4e703a9e,3414,-1,page_3_5,3404
2,content_9aa793d4d895,3707,2,page_3_5,3727
3,content_331d6c4de07b,580,-5,page_1,530
4,content_d99b7a2d90ca,2241,-8,page_3_5,2161


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
Explanation: Static if/else rules (e.g., if position > 10 and traffic_drop > 20%) are too rigid. They miss subtle non-linear interactions across seasonal trends, keyword difficulty, query intent shifts, and content age. ML models learn these complex multi-feature patterns dynamically and generalize far better to new data over time.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simple static rule: catch pages where position_tier is "10+" AND clicks dropped by > 5
simple_rule = df[(df['position_tier'] == '10+') & (df['clicks_drop'] > 5)]

# High-impact pages missed because they haven't slipped past page 1/2 yet
missed_opportunities = df[(df['clicks_drop'] > 10) & (df['position_tier'] != '10+')]

print(f"Pages caught by simple rule: {len(simple_rule)}")
print(f"High-drop pages missed by simple rule: {len(missed_opportunities)}")

Pages caught by simple rule: 0
High-drop pages missed by simple rule: 831


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.